# Temporary: Top-40 Tokens Per Market-Cap Class

This notebook prints top-40 next-token probabilities for one representative report in each market-cap class.

Use it to refine `market_cap_verbolizer` tokens and/or market-cap bucket definitions.


In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / 'src' / 'data_collection' / 'consts.py').is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher


/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Keep this prompt in sync with the best prompt from market_cap_prompt_search.
prompt = 'I think the market size of this company is:'

top_k_tokens = 40
random_state = 42

MC_LABELS = ['small', 'mid', 'large',]
MC_BINS = [-float('inf'), 2e9, 1e10, float('inf')]

# Current verbalizer baseline.
market_cap_verbolizer = {
    'micro': ['micro', 'tiny', 'microcap'],
    'small': ['small', 'smallcap'],
    'mid': ['mid', 'medium', 'midcap'],
    'large': ['large', 'big', 'largecap'],
    'mega': ['mega', 'giant', 'megacap'],
}

print('Market-cap classes:', ', '.join(MC_LABELS))
print('Prompt:', prompt)


Market-cap classes: small, mid, large
Prompt: I think the market size of this company is:


In [3]:
model_name = 'mistralai/Mistral-7B-v0.1'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    max_memory={0: '12GiB', 'cpu': '64GiB'},
)

model_driver = ModelDriver(model_name=model_name, model=model)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.12it/s]
2026-06-04 19:44:49,962 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
def bucket_market_cap(f_size: float) -> str | None:
    if pd.isna(f_size):
        return None
    return pd.cut(
        pd.Series([f_size]),
        bins=MC_BINS,
        labels=MC_LABELS,
        right=False,
    ).iloc[0]


fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=['raw_text', 'f_size'])
reports_df = fetcher._apply_company_filters(reports_df, {})
reports_df = reports_df[reports_df['raw_text'].notna()].copy()
reports_df['market_cap_label'] = reports_df['f_size'].apply(bucket_market_cap)
reports_df = reports_df[reports_df['market_cap_label'].notna()].copy()
reports_df['market_cap_label'] = reports_df['market_cap_label'].astype(str)

label_counts = reports_df['market_cap_label'].value_counts().reindex(MC_LABELS, fill_value=0)
display(label_counts)

if (label_counts == 0).any():
    missing = label_counts[label_counts == 0].index.tolist()
    raise ValueError(f'No rows available for market-cap classes: {missing}')

sample_parts = []
for label in MC_LABELS:
    part = reports_df[reports_df['market_cap_label'] == label].sample(n=4, random_state=random_state)
    sample_parts.append(part)

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sort_values('market_cap_label').reset_index(drop=True)

display(sample_df[['id', 'market_cap_label', 'ticker', 'f_size']])


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql_query(query, conn)


market_cap_label
small      171
mid       1876
large    11468
Name: count, dtype: int64

,id,market_cap_label,ticker,f_size
0,148025,large,META,1.430208e+12
1,85047,large,AWK,2.604036e+10
2,55262,large,BSX,5.041790e+10
3,152966,large,EIX,2.988906e+10
4,36523,mid,JNPR,7.025841e+09
5,8526,mid,AOS,9.288793e+09
6,40524,mid,EMN,7.215155e+09
7,4636,mid,ON,9.294112e+09
8,39359,small,EQT,1.287337e+09
9,30939,small,GEN,1.899369e+09


In [5]:
def get_top_tokens(text: str, prompt: str, k: int = 40) -> list[tuple[str, float]]:
    clean_text = model_driver.clean_report(text)

    prompt_tokens = model_driver.tokenizer(prompt, return_tensors='pt')['input_ids'].squeeze()
    if prompt_tokens.dim() == 0:
        prompt_tokens = prompt_tokens.unsqueeze(0)

    context_max_tokens = 8000
    max_report_tokens = max(1, context_max_tokens - prompt_tokens.shape[0])
    report_tokens = model_driver.tokenizer(
        clean_text,
        return_tensors='pt',
        truncation=True,
        max_length=max_report_tokens,
    )['input_ids'].squeeze()
    if report_tokens.dim() == 0:
        report_tokens = report_tokens.unsqueeze(0)

    tokens = torch.cat((report_tokens, prompt_tokens), dim=0).to(model_driver.device)
    token_prob_dict = model_driver.fast_inference(tokens)
    return sorted(token_prob_dict.items(), key=lambda x: x[1], reverse=True)[:k]


In [6]:
top_tokens_by_label = {}

for row in sample_df.itertuples(index=False):
    label = row.market_cap_label
    top_tokens = get_top_tokens(row.raw_text, prompt, k=top_k_tokens)
    top_tokens_by_label[label] = top_tokens

    print(f'\n{label}')
    print(f'Report id: {row.id} | ticker: {row.ticker} | f_size: {row.f_size}')
    print(f'Top {top_k_tokens} tokens by probability:')
    for token, prob in top_tokens:
        print(f'{token:<14} | {prob:.6f}')



large
Report id: 148025 | ticker: META | f_size: 1430207989243.0
Top 40 tokens by probability:
Ad             | 0.031738
Market         | 0.015015
The            | 0.013672
Small          | 0.012085
Large          | 0.012085
huge           | 0.010620
very           | 0.009094
Mark           | 0.008301
Very           | 0.007782
In             | 0.007538
Facebook       | 0.005890
Not            | 0.005707
Face           | 0.005035
Amaz           | 0.004883
How            | 0.004303
it             | 0.004028
Unknown        | 0.004028
advertising    | 0.004028
Re             | 0.003799
https          | 0.003677
What           | 0.003571
Mid            | 0.003342
Mass           | 0.003143
It             | 0.003052
About          | 0.002960
We             | 0.002869
Great          | 0.002869
Big            | 0.002869
Meta           | 0.002777
Acc            | 0.002609
Average        | 0.002609
Yes            | 0.002457
Too            | 0.002457
small          | 0.002380
Un             | 0.0